# Task 2, Part B3: Controlled Intervention (Delta-Rule State Updates) + Resource Costs

**Motivation (from B1/B2):** ELU+1 linear attention keeps a fixed d x d state per head but only ever *adds*
phi(k) v^T, so associations interfere as K grows (Zoology / MQAR). The window model cannot reach far queries.

**Intervention (one, chosen before evaluation):** *selective updates to the compressed recurrent state* -- the
delta rule. Each head reads its current prediction for k_t and writes only the correction:
S_t = S_{t-1} + beta_t k_t (v_t - S_{t-1}^T k_t)^T, beta_t = sigmoid(w_h^T x_t), with L2-normalised ELU+1 features.
* Comparator: the K=16 ELU+1 linear-attention model (same depth, width, heads, optimiser, 4,500 updates, 3 seeds).
* Persistent state per layer: delta 4 x 32 x 32 = 4,096 FP32 values vs linear 4 x (32 x 32 + 32) = 4,224 (the
  linear normaliser z is not needed by the delta read-out; no feature-dimension or precision change).
* Extra parameters: one 128 -> 4 gate per block (2 x 516 = 1,032), < 5% of the baseline (checked below).
* Development-selection rule: none -- no hyperparameter was tuned; the final (4,500-update) checkpoint is used and
  the frozen test sets are unchanged.

**Prediction (written before evaluation):** at K = 16 the delta-rule model will exceed ELU+1 linear attention on the
mixed regime at every distance bin (less interference), but remain below dense softmax attention, and its gain over
linear attention will be larger at K = 16 than any distance effect (state access is distance-independent).

In [ ]:
# --- Setup: locate (or clone) the PA4 repo, pin library versions, prepare data -------------
import os, sys, subprocess, importlib.metadata as md
REPO_URL = "https://github.com/moizejaz-31/ATML-PA4.git"   # change if your repo lives elsewhere

def _find_root(d):
    d = os.path.abspath(d)
    while True:
        if os.path.isdir(os.path.join(d, "shared")) and os.path.isdir(os.path.join(d, "task2_attention_kv")):
            return d
        if os.path.dirname(d) == d:
            return None
        d = os.path.dirname(d)

ROOT = _find_root(os.getcwd())
if ROOT is None:                                   # fresh Kaggle / Colab runtime
    subprocess.run(["git", "clone", "-q", REPO_URL, "ATML-PA4"], check=True)
    ROOT = os.path.abspath("ATML-PA4")
os.chdir(ROOT)
sys.path.insert(0, ROOT)

def _ver(p):
    try:
        return md.version(p)
    except md.PackageNotFoundError:
        return None

if _ver("transformers") != "4.57.1" or _ver("huggingface_hub") != "0.36.2":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==4.57.1", "huggingface_hub==0.36.2",
                    "datasets>=3,<5"], check=True)
    print(">>> Installed pinned versions. Restart the kernel, then run this cell again.")

from shared.src.nb import init_notebook
ctx = init_notebook("task2", need_public=False, need_prefix_banks=False)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from shared.src.io import load_json, save_json, read_jsonl
from shared.src.plotting import save_figure
from task2_attention_kv.src import mqar_data as D
from task2_attention_kv.src.mqar_train import PROTOCOL, train_run, test_eval, run_name
from task2_attention_kv.src.mqar_model import MQARModel, n_params, attended_pairs, state_bytes, inference_sweep

OUT = ctx.results / "mqar"
CACHE = ctx.checkpoints / "mqar_cache"
dev, test = D.load_frozen("development"), D.load_frozen("test")
SEEDS = PROTOCOL["seeds"]
p_lin, p_delta, p_dense = n_params(MQARModel("linear")), n_params(MQARModel("delta")), n_params(MQARModel("dense"))
print(f"params linear {p_lin:,} delta {p_delta:,} dense {p_dense:,} -> delta/linear - 1 = {p_delta/p_lin-1:.4%}")
assert abs(p_delta / p_lin - 1) < 0.05
save_json(OUT / "B3_intervention_spec.json", dict(intervention="delta-rule selective state update", comparator="linear_mixed_K16",
          params=dict(linear=p_lin, delta=p_delta, dense=p_dense), state_bytes=dict(linear=state_bytes("linear", 256), delta=state_bytes("delta", 256)),
          dev_selection="none (fixed design, final checkpoint)", prediction="delta > linear at K=16 mixed in every distance bin; < dense"))

In [ ]:
for seed in SEEDS:
    h = train_run("delta", 16, "mixed", seed, dev, OUT, CACHE)
    print(f"{h['name']}: {h['train_time_s']:.0f}s final dev {h['dev'][-1]}")
res = [test_eval(OUT, test, "delta", "mixed", 16, s) for s in SEEDS]

## Intervention vs comparator (K = 16, all seeds)

In [ ]:
ALL = [("dense", "mixed", 4), ("dense", "mixed", 8), ("dense", "mixed", 16), ("dense", "near", 16),
       ("window", "near", 4), ("window", "near", 8), ("window", "near", 16), ("window", "mixed", 16),
       ("linear", "mixed", 4), ("linear", "mixed", 8), ("linear", "mixed", 16), ("dilated", "mixed", 4), ("dilated", "mixed", 16),
       ("delta", "mixed", 16)]
rows = []
for kind, regime, k in ALL:
    for s in SEEDS:
        f = OUT / "test" / f"{run_name(kind, regime, k, s)}.json"
        if not f.exists():
            continue
        r = load_json(f)
        for cell, v in r["cells"].items():
            kk, reg = cell.split("_")
            rows.append(dict(model=kind, train=f"{regime} K{k}", seed=s, test_K=int(kk[1:]), test_regime=reg, **v))
for s in SEEDS:
    for regime in ("mixed", "near"):
        f = OUT / "test" / f"{run_name('dense', regime, 16, s)}@window.json"
        if f.exists():
            for cell, v in load_json(f)["cells"].items():
                kk, reg = cell.split("_")
                rows.append(dict(model="dense->window(test)", train=f"{regime} K16", seed=s, test_K=int(kk[1:]), test_regime=reg, **v))
df = pd.DataFrame(rows)
df.to_csv(OUT / "all_test_cells.csv", index=False)
bins = ["acc", "acc_d0_32", "acc_d32_64", "acc_d64_128", "acc_d128_256"]
k16 = df[(df.test_K == 16) & (df.test_regime == "mixed") & df.train.str.endswith("K16")]
k16.groupby(["model", "train"])[bins].agg(["mean", "std"]).round(3)

In [ ]:
from shared.src.stats import paired_bootstrap_diff
import gzip, json
def per_query(tag):
    with gzip.open(OUT / "test" / f"{tag}_queries.json.gz", "rt") as f:
        return np.array(json.load(f)["K16_mixed"]["correct"])
cmp = []
for s in SEEDS:
    a, b = per_query(run_name("delta", "mixed", 16, s)), per_query(run_name("linear", "mixed", 16, s))
    m, lo, hi = paired_bootstrap_diff(a, b)
    cmp.append(dict(seed=s, delta_acc=a.mean(), linear_acc=b.mean(), diff=m, ci_lo=lo, ci_hi=hi))
pd.DataFrame(cmp)

## Distance x load heatmaps (rows: model and its training K / regime; columns: test K x distance bin)

In [ ]:
bcols = ["acc_d0_32", "acc_d32_64", "acc_d64_128", "acc_d128_256"]
mixed = df[df.test_regime == "mixed"].groupby(["model", "train", "test_K"])[bcols].mean()
H = mixed.unstack("test_K")
H.columns = [f"K{k} {c.replace('acc_d', 'd')}" for c, k in H.columns]
H = H[[f"K{k} {c.replace('acc_d', 'd')}" for k in (4, 8, 16) for c in bcols]]
plt.figure(figsize=(16, 7))
sns.heatmap(H, annot=True, fmt=".2f", cmap="viridis", vmin=0, vmax=1, cbar_kws=dict(label="accuracy (mean of 3 seeds)"))
plt.title("MQAR accuracy by load K and source-query distance (mixed test regime); row = model [train regime, train K]")
save_figure(ctx.figures, "mqar_distance_by_load_heatmap")

## Theoretical costs at T = 256 vs measured forward-only inference sweep

In [ ]:
theo = pd.DataFrame([dict(kind=k, **attended_pairs(k, 256), **state_bytes(k, 256)) for k in ("dense", "window", "dilated", "linear", "delta")]).set_index("kind")
theo.to_csv(OUT / "theoretical_costs_T256.csv")
theo

In [ ]:
sweep_path = OUT / "inference_sweep.jsonl"
if not sweep_path.exists():
    from shared.src.io import write_jsonl
    rows = inference_sweep(["dense", "window", "window_banded", "dilated", "dilated_banded", "linear", "linear_prefix", "delta"])
    write_jsonl(sweep_path, rows)
sw = pd.DataFrame(read_jsonl(sweep_path))
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for kind, g in sw.groupby("kind"):
    g = g.dropna(subset=["ms"])
    axes[0].plot(g["T"], g["ms"], "o-", label=kind)
    axes[1].plot(g["T"], g["peak_allocated_mb"], "o-", label=kind)
for ax, t in zip(axes, ("forward latency (ms, B=4, 2 blocks, FP16)", "peak allocated (MiB)")):
    ax.set_xscale("log", base=2); ax.set_yscale("log"); ax.set_xlabel("T"); ax.set_title(t)
axes[0].legend(fontsize=8)
save_figure(ctx.figures, "mqar_inference_sweep")
sw.pivot_table(index=["kind", "impl"], columns="T", values="ms").round(2)

In [ ]:
tt = []
for kind, regime, k in ALL:
    H_ = [load_json(OUT / "histories" / f"{run_name(kind, regime, k, s)}.json") for s in SEEDS if (OUT / "histories" / f"{run_name(kind, regime, k, s)}.json").exists()]
    if H_:
        tt.append(dict(kind=kind, train=f"{regime} K{k}", params=H_[0]["n_params"], train_time_s=np.mean([h["train_time_s"] for h in H_]),
                       train_peak_mem_mb=np.mean([h["peak_mem_mb"] or 0 for h in H_])))
pd.DataFrame(tt)